# Interviewbot: *Harbour Lights* with Dana Lee

[ELIZA](https://en.wikipedia.org/wiki/ELIZA) (Joseph Weizenbaum, 1966) was one of the first chatbots. It played a psychotherapist using nothing more than **pattern matching and substitution**: it looked for keywords in what you typed, swapped pronouns ("my" becomes "your"), and slotted pieces of your sentence into a canned reply, usually a question.

This notebook builds an ELIZA-style **media interviewer** instead of a therapist. The bot is **Dana Lee**, host of the (fictional) radio show *Harbour Lights*, and **you** are the guest: the world's leading authority on **Hong Kong**. Dana asks about the city's food, neighbourhoods, history, language, films, festivals, transport and more.

## How it works

Like ELIZA, the approach is **rules + regular expressions**. The bot doesn't understand anything you say.

1. **Reflection:** a table swaps first and second person (`I → you`, `my → your`, `am → are`, ...), so your words can be echoed back as a question.
2. **Rules:** an ordered list of `(regex, response templates)` pairs, checked top to bottom. The first rule that matches wins, and its captured groups (reflected) fill `{0}`, `{1}` in a template. There are three tiers:
   - **Personal rules** (e.g. *"I grew up in ..."*, *"My favourite ... is ..."*)
   - **Hong Kong keyword rules** (dim sum, Star Ferry, Mong Kok, 1997, Wong Kar-wai, typhoons, ...)
   - **General ELIZA rules** (*"I love ..."*, *"I think ..."*, *"because ..."*, yes/no, questions)
3. **Memory:** like ELIZA's "my" memory, any *"my X"* phrase is saved and can come back later ("Earlier you mentioned your grandmother's shop...").
4. **Fallbacks and topic nudges:** if nothing matches, Dana gives a generic prompt, or after a few misses in a row she moves on to a new prepared Hong Kong question.
5. **Variety:** each rule works through its templates in shuffled order, so the same rule doesn't repeat itself right away.

In [1]:
import random
import re

# Pronoun reflection: turn the guest's words around so they can be echoed back.
REFLECTIONS = {
    "i": "you", "me": "you", "my": "your", "mine": "yours", "myself": "yourself",
    "am": "are", "i'm": "you're", "i've": "you've", "i'll": "you'll",
    "i'd": "you'd", "we": "you", "us": "you", "our": "your", "ours": "yours",
    "you": "I", "your": "my", "yours": "mine", "yourself": "myself",
    "you're": "I'm", "you've": "I've", "you'll": "I'll",
}

# Verbs that only change when they follow a pronoun: "I was" -> "you were", "you are" -> "I am",
# but "people are" and "it was" stay as they are.
VERB_REFLECTIONS = {
    ("i", "was"): "were", ("you", "are"): "am", ("you", "were"): "was",
    ("we", "are"): "are", ("we", "were"): "were",
}


def reflect(fragment):
    """Swap first/second person words, e.g. "my grandmother's shop" -> "your grandmother's shop"."""
    words = fragment.split()
    out = []
    for i, w in enumerate(words):
        prev = words[i - 1].lower() if i else ""
        out.append(VERB_REFLECTIONS.get((prev, w.lower())) or REFLECTIONS.get(w.lower(), w))
    return " ".join(out)


def clean(fragment):
    """Trim whitespace and trailing punctuation from a captured phrase."""
    return fragment.strip().rstrip(".!?,;:")


print(reflect("I think my city is the best"))
print(reflect("I was sure you are wrong, but people are friendly"))

you think your city is the best
you were sure I am wrong, but people are friendly


## The rules

Rules are checked **in order**, so the more specific ones come first. `{0}`, `{1}` are replaced by the reflected capture groups. Keyword rules capture the keyword itself so Dana can name it in the question.

In [2]:
# Tier 1: personal statements (checked before keywords so "I grew up in Mong Kok" is echoed back)
PERSONAL_RULES = [
    (r"\bmy fav(?:ou?rite)? (.+?) (?:is|are) (.+)",
     ["Why is {1} your favourite {0}?",
      "What makes {1} stand out above every other {0}?"]),
    (r"\bi (?:grew up|was born|was raised) in ([^,.;!?]+)",
     ["Growing up in {0}: what's a sound or smell that takes you straight back there?",
      "What did growing up in {0} teach you about Hong Kong that outsiders never learn?",
      "How has {0} changed since you were a child?"]),
    (r"\bi (?:live|lived|used to live) in ([^,.;!?]+)",
     ["What is daily life in {0} really like?",
      "If our listeners spent one day in {0}, where should they go first?"]),
    (r"\bi (?:first )?(?:visited|went to|moved to) ([^,.;!?]+)",
     ["What were your first impressions of {0}?",
      "What made you go to {0} in the first place?"]),
]

# Tier 2: Hong Kong keyword rules
HK_RULES = [
    (r"\b(dim sum|yum cha|har gow|siu mai|char siu bao|cheung fun)\b",
     ["Ah, {0}! As the leading authority, where would you take a first-timer for {0}?",
      "What separates truly great {0} from the ordinary kind?",
      "Is the tradition of {0} changing with the younger generation?"]),
    (r"\b(milk tea|egg tarts?|pineapple buns?|cha chaan tengs?|french toast|wonton noodles?|claypot rice|street food|curry fish balls?)\b",
     ["Our listeners love food talk. What makes Hong Kong {0} special?",
      "Where in the city would you go for the best {0}?",
      "How did {0} become such a part of everyday Hong Kong life?"]),
    (r"\b(victoria peak|the peak|peak tram)\b",
     ["Everyone photographs the view from {0}. What do they miss while they're up there?",
      "Is {0} worth the crowds, in your expert opinion?"]),
    (r"\b(victoria harbour|the harbour|harbor|star ferry|symphony of lights)\b",
     ["The {0} is almost a symbol of the city. What does it mean to you?",
      "How would you describe {0} to someone who has never seen it?"]),
    (r"\b(mtr|trams?|ding ding|minibus(?:es)?|octopus card|ferries|double-decker)\b",
     ["Hong Kong's transport is famous. What's the secret behind the {0}?",
      "Do you have a favourite route or story involving the {0}?"]),
    (r"\b(cantonese|english|mandarin|putonghua|language|slang)\b",
     ["How does {0} shape the identity of Hong Kong people?",
      "Could you teach our listeners one {0} expression every visitor should know?"]),
    (r"\b(repulse bay|tai tam)\b",
     ["Repulse Bay and Tai Tam are quite tucked away — what's it actually like living out in {0}?",
      "{0} feels a world apart from the rest of Hong Kong. What do you love most about that side of the island?",
      "How does living near {0} shape your day-to-day experience of Hong Kong?"]),
    (r"\b(hong kong international school|hkis)\b",
     ["What is it like growing up at {0} — straddling two cultures every single day?",
      "How did {0} shape the way you see Hong Kong differently from someone who went to a local school?",
      "Did going to {0} give you a unique perspective on the city?"]),
    (r"\b(mong kok|tsim sha tsui|central|kowloon|lantau|sheung wan|wan chai|causeway bay|sham shui po|stanley|new territories|sai kung|lamma|cheung chau|hong kong island)\b",
     ["{0}! What's something about {0} that only a local would know?",
      "How would you describe the character of {0} in a single sentence?",
      "If you had one afternoon in {0}, how would you spend it?"]),
    (r"\b(british|colonial|colony|1997|handover|opium war|history)\b",
     ["The {0} question is a big one. How does it still shape Hong Kong today?",
      "What's the most misunderstood part of Hong Kong's {0}?"]),
    (r"\b(bruce lee|jackie chan|wong kar-wai|stephen chow|canto-?pop|films?|movies?|cinema)\b",
     ["Hong Kong's cultural exports are legendary. Why did {0} have such global reach?",
      "What does {0} tell us about the spirit of the city?"]),
    (r"\b(typhoons?|weather|humidity|humid|rain|summer)\b",
     ["How do Hong Kongers live with the {0}?",
      "Any memorable {0} stories for our listeners?"]),
    (r"\b(lunar new year|chinese new year|mid-autumn|mooncakes?|dragon boats?|bun festival|festivals?)\b",
     ["Take us inside {0} in Hong Kong. What does it look and sound like?",
      "What's the one {0} tradition visitors should experience?"]),
    (r"\b(hiking|hikes?|dragon's back|lion rock|country parks?|beaches|mountains?|nature)\b",
     ["People are often surprised Hong Kong has so much {0}. Why is that side of the city overlooked?",
      "Where is your favourite spot for {0}, and why?"]),
    (r"\b(neon|skyline|skyscrapers?|buildings?|architecture|density)\b",
     ["The {0} is iconic. What does it say about how Hong Kong grew?",
      "Is the {0} disappearing, or evolving?"]),
    (r"\b(junk boats?|fishing villages?|tai o|stilt houses|temples?|walled villages?)\b",
     ["{0} shows a very different Hong Kong. Why should visitors seek it out?",
      "What's the story behind {0}?"]),
    (r"\b(protests?|politics|government|china|mainland)\b",
     ["That's a sensitive subject. How do you think {0} affects everyday life in Hong Kong?",
      "How would you explain {0} to someone outside Hong Kong?"]),
]

# Tier 3: general ELIZA-style conversational rules
GENERAL_RULES = [
    (r"\bi (?:love|adore|really like) (.+)",
     ["How long have you loved {0}?",
      "What is it about {0} that you love so much?"]),
    (r"\bi (?:hate|dislike|can't stand) (.+)",
     ["That's a strong opinion! What's wrong with {0}?",
      "Do most Hong Kongers share your feelings about {0}?"]),
    (r"\bi think (.+)",
     ["Why do you think {0}?",
      "Would other experts agree that {0}?"]),
    (r"\bi remember (.+)",
     ["What else comes to mind when you remember {0}?",
      "Why does that memory of {0} stay with you?"]),
    (r"\bi (?:don't|do not) (.+)",
     ["Why don't you {0}?",
      "Interesting. What would it take for you to {0}?"]),
    (r"\bi (?:feel|felt) (.+)",
     ["What makes you feel {0}?",
      "Do you often feel {0} when you talk about Hong Kong?"]),
    (r"\bi(?: am|'m) (.+)",
     ["Why do you say you're {0}?",
      "How does being {0} shape the way you see Hong Kong?"]),
    (r"\bbecause (.+)",
     ["Is that the only reason?",
      "And how does that connect to the Hong Kong you know?"]),
    (r"^(?:yes|yeah|yep|of course|definitely)\b",
     ["You sound certain. Can you give us an example?",
      "Tell us more."]),
    (r"^(?:no|nope|not really)\b",
     ["Why not?",
      "Fair enough. What would you say instead?"]),
    (r"(.+)\?$",
     ["I'm the one asking the questions today! But what's your take?",
      "Ha, turning the tables on me! What do you think?"]),
]

RULES = PERSONAL_RULES + HK_RULES + GENERAL_RULES

FALLBACKS = [
    "Fascinating. Could you elaborate for our listeners?",
    "Say more about that.",
    "And how does that fit into the bigger picture of Hong Kong?",
    "Interesting! Why do you say that?",
]

# Prepared questions Dana switches to when the conversation stalls
TOPIC_QUESTIONS = [
    "What's something about Hong Kong most people get wrong?",
    "If you had to describe Hong Kong in three words, which would you pick?",
    "What's the best meal you've ever had in Hong Kong?",
    "Which neighbourhood best captures the soul of the city?",
    "How has Hong Kong changed in the time you've been studying it?",
    "What's your favourite way to spend a Sunday in Hong Kong?",
    "Which Hong Kong film should everyone watch?",
    "What's the one place in Hong Kong you'd never tell tourists about?",
    "You grew up in Repulse Bay — how did that shape your love of Hong Kong?",
    "What was it like playing tennis in Causeway Bay as a kid?",
]

print(f"{len(RULES)} rules loaded.")

32 rules loaded.


## The interviewer

`InterviewBot.respond()` does the ELIZA loop for each line the guest types: record any *"my ..."* phrase in memory, try each rule in order, fill in the template with the reflected groups, and fall back to memory, a topic change, or a generic prompt if nothing matches.

In [3]:
MEMORY_PATTERN = re.compile(r"\bmy [^,.;!?]+", re.IGNORECASE)
EXIT_PATTERN = re.compile(r"^\s*(bye|goodbye|quit|exit|that's all|thanks,? bye)\b", re.IGNORECASE)


class InterviewBot:
    def __init__(self, host="Dana Lee", show="Harbour Lights", nudge_after=2):
        self.host = host
        self.show = show
        self.nudge_after = nudge_after       # fallbacks in a row before changing topic
        self.rules = [(re.compile(p, re.IGNORECASE), t) for p, t in RULES]
        self.queues = {}                      # rule index -> templates left to use
        self.memory = []                      # reflected "my ..." phrases
        self.misses = 0
        self.topics = random.sample(TOPIC_QUESTIONS, len(TOPIC_QUESTIONS))

    def greeting(self):
        return (f"Welcome to {self.show}, I'm {self.host}. Today's guest is the world's leading "
                f"authority on Hong Kong. Thank you for joining us! To start, how did you first "
                f"fall in love with Hong Kong?")

    def goodbye(self):
        return (f"That's all the time we have. Thank you for sharing your expertise on Hong Kong. "
                f"You've been listening to {self.show}; I'm {self.host}. Joi gin!")

    def _pick(self, key, templates):
        """Work through a rule's templates in shuffled order so it doesn't repeat itself too soon."""
        if not self.queues.get(key):
            self.queues[key] = random.sample(templates, len(templates))
        return self.queues[key].pop()

    def _remember(self, text):
        match = MEMORY_PATTERN.search(text)
        if match:
            self.memory.append(reflect(clean(match.group(0))))

    def respond(self, text):
        text = " ".join(text.split())
        if not text:
            return "Take your time. What's on your mind?"
        if EXIT_PATTERN.search(text):
            return None

        for i, (pattern, templates) in enumerate(self.rules):
            match = pattern.search(text)
            if match:
                self._remember(text)
                self.misses = 0
                groups = [reflect(clean(g)) for g in match.groups()]
                return self._pick(i, templates).format(*groups)

        # No rule matched: bring back a memory, change topic, or give a generic prompt
        self.misses += 1
        if self.memory and random.random() < 0.5:
            reply = f"Earlier you mentioned {self.memory.pop(0)}. Tell our listeners more about that."
        elif self.misses >= self.nudge_after and self.topics:
            self.misses = 0
            reply = "Let's change direction. " + self.topics.pop()
        else:
            reply = self._pick("fallback", FALLBACKS)
        self._remember(text)
        return reply

## Demo transcript

A scripted run so the bot can be seen without typing anything. The random seed is fixed so the output is reproducible.

In [4]:
random.seed(7)
bot = InterviewBot()

sample_inputs = [
    "I grew up in Mong Kok, above my grandmother's shop.",
    "Honestly, the best thing is the dim sum.",
    "I love the Star Ferry at night.",
    "I think Hong Kong is the most exciting city in Asia.",
    "It's hard to explain.",
    "Hmm.",
    "People forget how much hiking there is.",
    "The handover in 1997 changed everything.",
    "My favourite film is In the Mood for Love.",
    "What do you think?",
    "Goodbye!",
]

print(f"Interviewer: {bot.greeting()}\n")
for line in sample_inputs:
    print(f"Guest:       {line}")
    reply = bot.respond(line)
    if reply is None:
        print(f"Interviewer: {bot.goodbye()}")
        break
    print(f"Interviewer: {reply}\n")

Interviewer: Welcome to Harbour Lights, I'm Dana Lee. Today's guest is the world's leading authority on Hong Kong. Thank you for joining us! To start, how did you first fall in love with Hong Kong?

Guest:       I grew up in Mong Kok, above my grandmother's shop.
Interviewer: What did growing up in Mong Kok teach you about Hong Kong that outsiders never learn?

Guest:       Honestly, the best thing is the dim sum.
Interviewer: Is the tradition of dim sum changing with the younger generation?

Guest:       I love the Star Ferry at night.
Interviewer: How would you describe Star Ferry to someone who has never seen it?

Guest:       I think Hong Kong is the most exciting city in Asia.
Interviewer: Would other experts agree that Hong Kong is the most exciting city in Asia?

Guest:       It's hard to explain.
Interviewer: Interesting! Why do you say that?

Guest:       Hmm.
Interviewer: Let's change direction. You grew up in Repulse Bay — how did that shape your love of Hong Kong?

Guest:  

## Your turn: live interview

Run the cell below and answer Dana's questions. Type **bye** (or *quit* / *exit*) to end the interview.

In [ ]:
def chat():
    bot = InterviewBot()
    print(f"Interviewer: {bot.greeting()}")
    while True:
        try:
            line = input("You: ")
        except (EOFError, KeyboardInterrupt):
            line = "bye"
        reply = bot.respond(line)
        if reply is None:
            print(f"Interviewer: {bot.goodbye()}")
            break
        print(f"Interviewer: {reply}")


chat()

## Use of Generative AI

This notebook was written with **Claude Code** (Anthropic's coding assistant, model Claude Opus 5.5).

**Prompts used**

1. The full text of the assignment ("Problem 2: Code Interviewbot", starting *"ELIZA was one of the earliest natural language processing programs..."*), followed by `/plan`, which asked Claude to put together an implementation plan before writing any code.
2. Claude asked two clarifying questions, and I answered:
   - *What topic should the interviewer ask you about?* → **"Hong Kong"**
   - *Which chatbot approach should the notebook use?* → **"Rules + regex"**
3. I approved the plan Claude proposed.
4. *"is this assignment ready for submission"*: Claude reviewed the notebook and pointed out that some echoed replies were ungrammatical (e.g. "people **am** friendly") and that the notebook had no saved outputs.
5. *"ok make the fixes and save in jupyter"*: Claude fixed the pronoun reflection so verbs only change after a pronoun, reworded an awkward template, added an "I am ..." rule, and executed the notebook in Jupyter to save the outputs.
6. *"add Repulse Bay (my neighbourhood), tai tam and Hong Kong International School (my school), causeway bay (neighbourhood I played tennis in)"*: I added the keyword rules and question templates for those personal locations myself.

**How the AI contributed**

- **Design:** I chose the topic (Hong Kong) and the rules-based regex approach. Claude designed the specific implementation: a reflection table, three ordered tiers of regex rules, ELIZA-style "my ..." memory, and fallback topic nudges.
- **Code:** Claude wrote the Python code in this notebook, including the base rule set.
- **Testing:** Claude ran the code cells to check that the rules fire as expected (keyword matches, pronoun reflection, memory recall, exit handling).
- **My role:** I chose the topic and approach, reviewed and approved the plan, reviewed the code, and ran and tested the notebook. I added the Hong Kong keyword rules and question templates for my own neighbourhoods and school (Repulse Bay, Tai Tam, Causeway Bay, Hong Kong International School) and the related topic questions.